Prepares the textual and categorical information that will be used in the clustering stage.

Focus on the core content fields of each film:
- plotMedium
- plotShort (as fallback)
- genres
- keywords
- mainCountry
- firstLanguage

The goal of this notebook is to build a clean baseline feature representation, with emphasis on narrative text.

In [1]:
import ast
import re
from collections import Counter
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, hstack, save_npz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text

In [2]:
PROJECT_ROOT = Path.cwd().parents[1]

DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs" / "text_clustering"
FEATURES_DIR = OUTPUTS_DIR / "features"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "European_data_2000.csv"

CORE_COLUMNS = [
    "title",
    "plotMedium",
    "plotShort",
    "genres",
    "keywords",
    "mainCountry",
    "firstLanguage",
]

In [3]:
def load_dataset(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")


print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Exists:", DATA_PATH.exists())

df = load_dataset(DATA_PATH)

print("Original shape:", df.shape)
print("Columns:", list(df.columns))

Current working directory: c:\Users\Georg\Desktop\DTU\FATA\FataProject\notebooks\text-clustering
Project root: c:\Users\Georg\Desktop\DTU\FATA\FataProject
Data path: c:\Users\Georg\Desktop\DTU\FATA\FataProject\data\European_data_2000.csv
Exists: True
Original shape: (2000, 23)
Columns: ['titleId', 'imdbUrl', 'originalTitle', 'titleType', 'releaseYear', 'runtimeMinutes', 'isAdult', 'imdbRating', 'numberOfVotes', 'allCountries', 'mainCountry', 'allLanguages', 'firstLanguage', 'englishTitle', 'topFiveActors', 'directors', 'writers', 'plotShort', 'plotMedium', 'plotLong', 'genres', 'keywords', 'production']


In [4]:
available_core_columns = [col for col in CORE_COLUMNS if col in df.columns]
missing_core_columns = [col for col in CORE_COLUMNS if col not in df.columns]

print("Available core columns:", available_core_columns)
print("Missing core columns:", missing_core_columns)

df = df[available_core_columns].copy()

print("Selected shape:", df.shape)
df.head(3)

Available core columns: ['plotMedium', 'plotShort', 'genres', 'keywords', 'mainCountry', 'firstLanguage']
Missing core columns: ['title']
Selected shape: (2000, 6)


,plotMedium,plotShort,genres,keywords,mainCountry,firstLanguage
0,"Poland, 1962. Anna, an orphan brought up by nu...",A novice nun about to take her vows uncovers a...,Drama,"poland, 1960s, jew, novitiate, family secret, ...",PL,pl
1,Loosely-based on the true events of a homeless...,A homeless loner moves into a self-storage fac...,"Horror, Thriller","stalker, female nudity, female topless nudity,...",GB,en
2,"Louis, a young teacher enamored of the age of ...",A man who escorts wealthy widows in New York's...,Comedy,"new york city, brassiere, painting oneself, ca...",FR,en


In [5]:
missing_summary = df.isna().sum().sort_values(ascending=False).to_frame("missing_values")
missing_summary["missing_pct"] = (missing_summary["missing_values"] / len(df) * 100).round(2)
missing_summary

,missing_values,missing_pct
plotMedium,0,0.0
plotShort,0,0.0
genres,0,0.0
keywords,0,0.0
mainCountry,0,0.0
firstLanguage,0,0.0


## 2. Build the narrative text field

The main narrative field is `plotMedium`, because it is informative without being too long or too noisy.

When `plotMedium` is missing, we use `plotShort` as a fallback.

In [6]:
def normalize_text_input(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


if "plotMedium" not in df.columns:
    df["plotMedium"] = ""

if "plotShort" not in df.columns:
    df["plotShort"] = ""

df["plotMedium"] = df["plotMedium"].apply(normalize_text_input)
df["plotShort"] = df["plotShort"].apply(normalize_text_input)

# Use plotMedium as the primary narrative field and plotShort as fallback.
df["plot_final"] = np.where(
    df["plotMedium"].str.len() > 0,
    df["plotMedium"],
    df["plotShort"]
)

df["plot_source"] = np.where(
    df["plotMedium"].str.len() > 0,
    "plotMedium",
    np.where(df["plotShort"].str.len() > 0, "plotShort", "missing")
)

df[["plotMedium", "plotShort", "plot_final", "plot_source"]].head(5)

,plotMedium,plotShort,plot_final,plot_source
0,"Poland, 1962. Anna, an orphan brought up by nu...",A novice nun about to take her vows uncovers a...,"Poland, 1962. Anna, an orphan brought up by nu...",plotMedium
1,Loosely-based on the true events of a homeless...,A homeless loner moves into a self-storage fac...,Loosely-based on the true events of a homeless...,plotMedium
2,"Louis, a young teacher enamored of the age of ...",A man who escorts wealthy widows in New York's...,"Louis, a young teacher enamored of the age of ...",plotMedium
3,"An outlandish science-fiction, comedy-thriller...","In this spoof of spy films, CIA agent, Kelly, ...","An outlandish science-fiction, comedy-thriller...",plotMedium
4,"Mrs. Palfrey (Dame Joan Plowright), recently w...",All but abandoned by her family in a London re...,"Mrs. Palfrey (Dame Joan Plowright), recently w...",plotMedium


In [7]:
df["has_plot"] = df["plot_final"].str.strip().str.len() > 0

print("Films with usable plot text:", int(df["has_plot"].sum()))
print("Films without usable plot text:", int((~df["has_plot"]).sum()))

df["plot_source"].value_counts(dropna=False)

Films with usable plot text: 2000
Films without usable plot text: 0


plot_source
plotMedium    2000
Name: count, dtype: int64

## 3. Clean the narrative text

The text is cleaned in a conservative way:
- lowercase conversion
- removal of unnecessary punctuation and symbols
- normalization of whitespace

The goal is to reduce noise without damaging meaning.

In [8]:
def clean_plot_text(text: str) -> str:
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Keep the cleaning simple so the baseline remains easy to interpret.
df["plot_clean"] = df["plot_final"].apply(clean_plot_text)

df[["plot_final", "plot_clean"]].head(3)

,plot_final,plot_clean
0,"Poland, 1962. Anna, an orphan brought up by nu...",poland 1962 anna an orphan brought up by nuns ...
1,Loosely-based on the true events of a homeless...,loosely based on the true events of a homeless...
2,"Louis, a young teacher enamored of the age of ...",louis a young teacher enamored of the age of f...


## 4. Prepare editorial tags

Genres and keywords are treated as curated tags rather than free narrative text.

For this reason:
- genres are converted into multi-label format
- keywords are cleaned and filtered
- rare keywords can be removed to reduce sparsity

In [9]:
def parse_list_like_column(value):
    if pd.isna(value):
        return []

    if isinstance(value, list):
        items = value
    else:
        text = str(value).strip()

        if text == "":
            return []

        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                items = parsed
            else:
                items = text.split(",")
        except (ValueError, SyntaxError):
            items = text.split(",")

    cleaned_items = []
    for item in items:
        item = str(item).strip().lower()
        item = re.sub(r"\s+", " ", item)
        if item and item not in {"nan", "none", "null"}:
            cleaned_items.append(item)

    return cleaned_items

In [10]:
if "genres" not in df.columns:
    df["genres"] = ""

if "keywords" not in df.columns:
    df["keywords"] = ""

df["genres_list"] = df["genres"].apply(parse_list_like_column)
df["keywords_list"] = df["keywords"].apply(parse_list_like_column)

df[["genres", "genres_list", "keywords", "keywords_list"]].head(5)

,genres,genres_list,keywords,keywords_list
0,Drama,[drama],"poland, 1960s, jew, novitiate, family secret, ...","[poland, 1960s, jew, novitiate, family secret,..."
1,"Horror, Thriller","[horror, thriller]","stalker, female nudity, female topless nudity,...","[stalker, female nudity, female topless nudity..."
2,Comedy,[comedy],"new york city, brassiere, painting oneself, ca...","[new york city, brassiere, painting oneself, c..."
3,"Action, Comedy","[action, comedy]","espionage, american agent, female british spy,...","[espionage, american agent, female british spy..."
4,"Comedy, Drama","[comedy, drama]","hotel, writer, grandson, friendship, impersona...","[hotel, writer, grandson, friendship, imperson..."


In [11]:
genre_counter = Counter()
for items in df["genres_list"]:
    genre_counter.update(items)

genre_freq = pd.DataFrame(
    genre_counter.items(),
    columns=["genre", "count"]
).sort_values("count", ascending=False).reset_index(drop=True)

genre_freq.head(20)

,genre,count
0,drama,1188
1,thriller,628
2,comedy,530
3,romance,413
4,horror,405
5,adventure,333
6,crime,331
7,action,310
8,mystery,285
9,fantasy,221


In [12]:
keyword_counter = Counter()
for items in df["keywords_list"]:
    keyword_counter.update(items)

keyword_freq = pd.DataFrame(
    keyword_counter.items(),
    columns=["keyword", "count"]
).sort_values("count", ascending=False).reset_index(drop=True)

keyword_freq.head(30)

,keyword,count
0,murder,682
1,death,603
2,female nudity,586
3,blood,550
4,husband wife relationship,528
5,violence,496
6,flashback,463
7,cigarette smoking,434
8,kiss,419
9,based on novel,413


In [13]:
MIN_KEYWORD_FREQ = 5

valid_keywords = {
    keyword
    for keyword, count in keyword_counter.items()
    if count >= MIN_KEYWORD_FREQ
}

# Rare keywords are removed to reduce noise and extreme sparsity.
df["keywords_filtered"] = df["keywords_list"].apply(
    lambda items: [item for item in items if item in valid_keywords]
)

print("Original unique keywords:", len(keyword_counter))
print("Filtered unique keywords:", len(valid_keywords))

df[["keywords_list", "keywords_filtered"]].head(5)

Original unique keywords: 56092
Filtered unique keywords: 9327


,keywords_list,keywords_filtered
0,"[poland, 1960s, jew, novitiate, family secret,...","[poland, 1960s, jew, family secret, loss of vi..."
1,"[stalker, female nudity, female topless nudity...","[stalker, female nudity, female topless nudity..."
2,"[new york city, brassiere, painting oneself, c...","[new york city, brassiere, car, barbershop, ch..."
3,"[espionage, american agent, female british spy...","[espionage, amazon river, yacht, orchid, helic..."
4,"[hotel, writer, grandson, friendship, imperson...","[hotel, writer, friendship, impersonation, wid..."


## 5. Prepare market context variables

Country and language are kept as categorical context variables.

They are not the main signal of similarity, but they can help capture production and market context.

In [14]:
def clean_category_value(value, default="unknown"):
    if pd.isna(value):
        return default

    value = str(value).strip().lower()
    value = re.sub(r"\s+", " ", value)

    if value == "":
        return default

    return value

In [15]:
if "mainCountry" not in df.columns:
    df["mainCountry"] = ""

if "firstLanguage" not in df.columns:
    df["firstLanguage"] = ""

df["mainCountry_clean"] = df["mainCountry"].apply(clean_category_value)
df["firstLanguage_clean"] = df["firstLanguage"].apply(clean_category_value)

df[["mainCountry", "mainCountry_clean", "firstLanguage", "firstLanguage_clean"]].head(5)

,mainCountry,mainCountry_clean,firstLanguage,firstLanguage_clean
0,PL,pl,pl,pl
1,GB,gb,en,en
2,FR,fr,en,en
3,IT,it,en,en
4,GB,gb,en,en


## 6. Build the narrative feature space with TF-IDF

The main feature representation of the baseline model is TF-IDF on the cleaned plot text.

This allows us to convert each film plot into a weighted numerical representation, where informative words receive more importance than very common words.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text

TFIDF_MAX_FEATURES = 4000
TFIDF_MIN_DF = 5
TFIDF_MAX_DF = 0.60

custom_stopwords = {
    "man", "woman", "young", "new", "life", "world", "time", "story", "family", "love"
}

all_stopwords = text.ENGLISH_STOP_WORDS.union(custom_stopwords)

tfidf_vectorizer = TfidfVectorizer(
    stop_words=list(all_stopwords),
    max_features=TFIDF_MAX_FEATURES,
    min_df=TFIDF_MIN_DF,
    max_df=TFIDF_MAX_DF,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b"
)

X_plot_tfidf = tfidf_vectorizer.fit_transform(df["plot_clean"])

feature_names = tfidf_vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", X_plot_tfidf.shape)
print("Number of TF-IDF terms:", len(feature_names))
print("Sample terms:")
print(feature_names[:50])

feature_name_set = set(feature_names)
for word in ["life", "love", "man", "young", "world", "time", "family", "story"]:
    print(word, word in feature_name_set)

TF-IDF matrix shape: (2000, 3946)
Number of TF-IDF terms: 3946
Sample terms:
['abandon' 'abandoned' 'abby' 'abducted' 'abilities' 'ability' 'able'
 'aboard' 'abruptly' 'absence' 'absolutely' 'abuse' 'abused' 'abusive'
 'academic' 'academy' 'accept' 'acceptance' 'accepted' 'accepts' 'access'
 'accident' 'accidental' 'accidentally' 'acclaimed' 'accompanied'
 'accomplish' 'according' 'account' 'accused' 'accustomed' 'achieve'
 'acquaintance' 'acquaintances' 'act' 'acting' 'action' 'actions' 'active'
 'activist' 'activities' 'actor' 'actors' 'actress' 'acts' 'actual'
 'actually' 'ad' 'adam' 'adaptation']
life False
love False
man False
young False
world False
time False
family False
story False


In [17]:
feature_names = tfidf_vectorizer.get_feature_names_out()

print("Number of TF-IDF terms:", len(feature_names))
print("Sample terms:")
print(feature_names[:50])

Number of TF-IDF terms: 3946
Sample terms:
['abandon' 'abandoned' 'abby' 'abducted' 'abilities' 'ability' 'able'
 'aboard' 'abruptly' 'absence' 'absolutely' 'abuse' 'abused' 'abusive'
 'academic' 'academy' 'accept' 'acceptance' 'accepted' 'accepts' 'access'
 'accident' 'accidental' 'accidentally' 'acclaimed' 'accompanied'
 'accomplish' 'according' 'account' 'accused' 'accustomed' 'achieve'
 'acquaintance' 'acquaintances' 'act' 'acting' 'action' 'actions' 'active'
 'activist' 'activities' 'actor' 'actors' 'actress' 'acts' 'actual'
 'actually' 'ad' 'adam' 'adaptation']


## 7. Encode the non-text content

The baseline representation combines:
- TF-IDF for plot text
- multi-hot encoding for genres
- binary encoding for filtered keywords
- one-hot encoding for country and language

This creates a hybrid feature space that combines story content, editorial labels, and production context.

In [18]:
genre_mlb = MultiLabelBinarizer()
X_genres = genre_mlb.fit_transform(df["genres_list"])
X_genres = csr_matrix(X_genres)

print("Genres matrix shape:", X_genres.shape)
print("Number of genre features:", len(genre_mlb.classes_))

Genres matrix shape: (2000, 24)
Number of genre features: 24


In [19]:
keyword_mlb = MultiLabelBinarizer()
X_keywords = keyword_mlb.fit_transform(df["keywords_filtered"])
X_keywords = csr_matrix(X_keywords)

print("Keywords matrix shape:", X_keywords.shape)
print("Number of keyword features:", len(keyword_mlb.classes_))

Keywords matrix shape: (2000, 9327)
Number of keyword features: 9327


In [20]:
metadata_df = df[["mainCountry_clean", "firstLanguage_clean"]].copy()

try:
    metadata_ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    metadata_ohe = OneHotEncoder(handle_unknown="ignore", sparse=True)

X_metadata = metadata_ohe.fit_transform(metadata_df)

print("Metadata matrix shape:", X_metadata.shape)

Metadata matrix shape: (2000, 74)


In [21]:
# Keep both a plot-only matrix and a full hybrid matrix for later comparison.
X_plot_only = X_plot_tfidf
X_full_baseline = hstack([X_plot_tfidf, X_genres, X_keywords, X_metadata]).tocsr()

print("Plot-only matrix shape:", X_plot_only.shape)
print("Full baseline matrix shape:", X_full_baseline.shape)

Plot-only matrix shape: (2000, 3946)
Full baseline matrix shape: (2000, 13371)


## 8. Sanity checks

Before saving the outputs, we check:
- the number of films
- the number of features in each block
- the final matrix shape

This ensures that the baseline feature space is complete and ready for the clustering stage.

In [22]:
n_tfidf_features = int(X_plot_tfidf.shape[1])
n_genre_features = int(X_genres.shape[1])
n_keyword_features = int(X_keywords.shape[1])
n_metadata_features = int(X_metadata.shape[1])
n_full_features = int(X_full_baseline.shape[1])

summary_rows = [
    ("Number of films", len(df)),
    ("Films with usable plot", int(df["has_plot"].sum())),
    ("TF-IDF features", n_tfidf_features),
    ("Genre features", n_genre_features),
    ("Keyword features", n_keyword_features),
    ("Metadata features", n_metadata_features),
    ("Full baseline features", n_full_features),
]

summary_df = pd.DataFrame(summary_rows, columns=["metric", "value"])
summary_df

,metric,value
0,Number of films,2000
1,Films with usable plot,2000
2,TF-IDF features,3946
3,Genre features,24
4,Keyword features,9327
5,Metadata features,74
6,Full baseline features,13371


In [23]:
processed_output_path = FEATURES_DIR / "processed_films.csv"

df.to_csv(processed_output_path, index=False)

joblib.dump(tfidf_vectorizer, FEATURES_DIR / "tfidf_vectorizer.joblib")
joblib.dump(genre_mlb, FEATURES_DIR / "genre_mlb.joblib")
joblib.dump(keyword_mlb, FEATURES_DIR / "keyword_mlb.joblib")
joblib.dump(metadata_ohe, FEATURES_DIR / "metadata_ohe.joblib")

save_npz(FEATURES_DIR / "X_plot_tfidf.npz", X_plot_tfidf)
save_npz(FEATURES_DIR / "X_genres.npz", X_genres)
save_npz(FEATURES_DIR / "X_keywords.npz", X_keywords)
save_npz(FEATURES_DIR / "X_metadata.npz", X_metadata)
save_npz(FEATURES_DIR / "X_plot_only.npz", X_plot_only)
save_npz(FEATURES_DIR / "X_full_baseline.npz", X_full_baseline)

print("Saved all feature outputs to:", FEATURES_DIR)

Saved all feature outputs to: c:\Users\Georg\Desktop\DTU\FATA\FataProject\outputs\text_clustering\features
